In [3]:
# ============================================================
# 0. Colab setup
# Run this first in Google Colab.
# ============================================================

!pip -q install -U yfinance numpy pandas scipy matplotlib

ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'c:\\users\\ws20w\\appdata\\local\\packages\\pythonsoftwarefoundation.python.3.11_qbz5n2kfra8p0\\localcache\\local-packages\\python311\\site-packages\\cffi-1.16.0.dist-info\\'


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\ws20w\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Tuple

import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf


# ============================================================
# SciPy-free replacements
# ============================================================

class _Norm:
    """Minimal replacement for scipy.stats.norm.cdf."""
    @staticmethod
    def cdf(x):
        arr = np.asarray(x, dtype=float)

        def _scalar_cdf(z):
            return 0.5 * (1.0 + math.erf(float(z) / math.sqrt(2.0)))

        out = np.vectorize(_scalar_cdf)(arr)

        if out.ndim == 0:
            return float(out)

        return out


norm = _Norm()


def gamma(x):
    """Minimal replacement for scipy.special.gamma."""
    return math.gamma(float(x))


def solve_banded(l_and_u, ab, b):
    """
    Minimal replacement for scipy.linalg.solve_banded for tridiagonal systems only.

    This code only supports the case used in our TFBS solver:
        solve_banded((1, 1), ab, rhs)

    ab format:
        ab[0, 1:]  = upper diagonal
        ab[1, :]   = main diagonal
        ab[2, :-1] = lower diagonal
    """
    if tuple(l_and_u) != (1, 1):
        raise NotImplementedError("This replacement only supports tridiagonal solve_banded((1, 1), ab, b).")

    ab = np.asarray(ab, dtype=float)
    d = np.asarray(b, dtype=float).copy()

    if ab.ndim != 2 or ab.shape[0] != 3:
        raise ValueError("ab must have shape (3, n).")

    n = ab.shape[1]

    if d.shape[0] != n:
        raise ValueError("b length must match ab.shape[1].")

    lower = ab[2, :-1].copy()
    diag = ab[1, :].copy()
    upper = ab[0, 1:].copy()

    if n == 1:
        if abs(diag[0]) < 1e-14:
            raise np.linalg.LinAlgError("Singular matrix in tridiagonal solver.")
        return d / diag[0]

    # Forward elimination
    for i in range(1, n):
        pivot = diag[i - 1]

        if abs(pivot) < 1e-14:
            raise np.linalg.LinAlgError("Near-zero pivot in tridiagonal solver.")

        factor = lower[i - 1] / pivot
        diag[i] = diag[i] - factor * upper[i - 1]
        d[i] = d[i] - factor * d[i - 1]

    # Back substitution
    x = np.empty(n, dtype=float)

    if abs(diag[-1]) < 1e-14:
        raise np.linalg.LinAlgError("Near-zero final pivot in tridiagonal solver.")

    x[-1] = d[-1] / diag[-1]

    for i in range(n - 2, -1, -1):
        if abs(diag[i]) < 1e-14:
            raise np.linalg.LinAlgError("Near-zero pivot in tridiagonal solver.")

        x[i] = (d[i] - upper[i] * x[i + 1]) / diag[i]

    return x


print("SciPy-free imports OK")
print("norm.cdf(0) =", norm.cdf(0))
print("gamma(1)   =", gamma(1))

ImportError: C extension: pandas.util not built. If you want to import pandas from the source directory, you may need to run 'python -m pip install -ve . --no-build-isolation -Ceditable-verbose=true' to build the C extensions first.

In [ ]:
# ============================================================
# Config
# ============================================================

@dataclass
class MarketDataConfig:
    ticker: str = "AAPL"
    period: str = "1y"

    # "mid"  : only valid bid-ask midpoint
    # "last" : last transaction price
    # "auto" : use mid if valid, otherwise last
    price_source: str = "auto"

    # Balanced maturity selection
    # If target_dtes is not None, choose expiries closest to these DTEs.
    target_dtes: Optional[Tuple[int, ...]] = (14, 30, 60, 90, 120, 180, 270, 360)
    max_expiries: Optional[int] = None

    max_strikes_per_expiry: int = 8
    min_dte: int = 0
    max_dte: Optional[int] = 365

    moneyness_min: float = 0.70
    moneyness_max: float = 1.30

    max_spread_pct: Optional[float] = 0.50
    min_volume: Optional[int] = None
    min_open_interest: Optional[int] = None

    data_debug: bool = True


@dataclass
class ExperimentConfig:
    r: float = 0.0426
    q: float = 0.0

    alpha_grid_coarse: Tuple[float, ...] = (0.50, 0.75)
    sigma_grid_coarse: Tuple[float, ...] = (0.20, 0.30, 0.40)
    lambda_grid_coarse: Tuple[float, ...] = (0.0, 4.0, 8.0)

    m_scan: int = 25
    n_scan: int = 25

    m_final_coarse: int = 35
    n_final_coarse: int = 35
    m_final: int = 50
    n_final: int = 50

    refine_alpha: bool = True
    alpha_radius: float = 0.10
    alpha_step: float = 0.05

    sigma_radius: float = 0.05
    sigma_step: float = 0.025
    sigma_bounds: Tuple[float, float] = (0.03, 1.00)

    lambda_radius: float = 4.0
    lambda_step: float = 2.0
    lambda_bounds: Tuple[float, float] = (0.0, 30.0)

    smax_mult: float = 3.0

    short_days: int = 30
    medium_days: int = 180

    progress: bool = True


# ============================================================
# Utilities
# ============================================================

def _as_float_array(x: Iterable[float]) -> np.ndarray:
    arr = np.asarray(list(x), dtype=float)
    return arr[np.isfinite(arr)]


def local_grid(center: float, radius: float, step: float, lower: float, upper: float) -> np.ndarray:
    if center is None or not np.isfinite(center):
        return np.array([], dtype=float)

    lo = max(float(lower), float(center) - float(radius))
    hi = min(float(upper), float(center) + float(radius))

    if lo > hi:
        return np.array([], dtype=float)

    grid = np.arange(lo, hi + 0.5 * step, step, dtype=float)
    grid = grid[(grid >= lower - 1e-12) & (grid <= upper + 1e-12)]

    return np.unique(np.round(grid, 10))


def pooled_rmse_from_sse(df: pd.DataFrame, sse_col: str, n_col: str) -> float:
    if df is None or df.empty:
        return np.nan

    total_sse = float(df[sse_col].sum())
    total_n = int(df[n_col].sum())

    if total_n <= 0 or not np.isfinite(total_sse):
        return np.nan

    return float(np.sqrt(total_sse / total_n))


def add_maturity_bucket(
    df: pd.DataFrame,
    short_days: int = 30,
    medium_days: int = 180,
    t_col: str = "T",
) -> pd.DataFrame:
    out = df.copy()
    days = 365.25 * out[t_col].astype(float)

    out["T_days"] = days
    out["bucket"] = np.select(
        [days <= short_days, days <= medium_days],
        ["Short", "Medium"],
        default="Long",
    )

    return out


def _select_expiries_balanced(
    expiry_rows: List[Tuple[str, int, float]],
    target_dtes: Optional[Tuple[int, ...]],
    max_expiries: Optional[int],
) -> List[Tuple[str, int, float]]:
    """
    expiry_rows: list of (expiry_str, dte, T)
    """
    expiry_rows = sorted(expiry_rows, key=lambda x: x[1])

    if target_dtes is None:
        if max_expiries is not None:
            return expiry_rows[: int(max_expiries)]
        return expiry_rows

    selected: Dict[str, Tuple[str, int, float]] = {}

    for target in target_dtes:
        if len(expiry_rows) == 0:
            break

        best = min(expiry_rows, key=lambda x: abs(x[1] - target))
        selected[best[0]] = best

    selected_rows = sorted(selected.values(), key=lambda x: x[1])

    if max_expiries is not None:
        selected_rows = selected_rows[: int(max_expiries)]

    return selected_rows


# ============================================================
# Data collection
# ============================================================

def collect_call_options_yfinance(cfg: MarketDataConfig) -> Tuple[float, pd.DataFrame]:
    if cfg.price_source not in {"auto", "mid", "last"}:
        raise ValueError("price_source must be one of: 'auto', 'mid', 'last'")

    ticker_obj = yf.Ticker(cfg.ticker)

    hist = ticker_obj.history(period=cfg.period, auto_adjust=False)

    if hist.empty:
        raise RuntimeError(f"No price history returned for {cfg.ticker}")

    S0 = float(hist["Close"].dropna().iloc[-1])

    expiries = list(ticker_obj.options)

    if len(expiries) == 0:
        raise RuntimeError(f"No option expiries returned for {cfg.ticker}")

    today = pd.Timestamp.utcnow().normalize().tz_localize(None)

    expiry_rows = []

    for exp in expiries:
        exp_ts = pd.Timestamp(exp)
        dte = int((exp_ts - today).days)

        if dte < cfg.min_dte:
            continue
        if cfg.max_dte is not None and dte > cfg.max_dte:
            continue

        expiry_rows.append((exp, dte, dte / 365.25))

    expiry_rows = _select_expiries_balanced(
        expiry_rows,
        target_dtes=cfg.target_dtes,
        max_expiries=cfg.max_expiries,
    )

    if len(expiry_rows) == 0:
        raise RuntimeError("No expiries survived DTE filtering.")

    rows = []
    debug_rows = []

    for exp, dte, T in expiry_rows:
        report = {
            "expiry": exp,
            "dte": dte,
            "raw": 0,
            "moneyness": 0,
            "valid_mid": 0,
            "valid_last": 0,
            "chosen_price": 0,
            "after_liquidity": 0,
            "selected": 0,
        }

        try:
            chain = ticker_obj.option_chain(exp)
            calls = chain.calls.copy()
        except Exception as exc:
            report["error"] = str(exc)
            debug_rows.append(report)
            print(f"[warning] failed option_chain({exp}): {exc}")
            continue

        report["raw"] = len(calls)

        if calls.empty:
            debug_rows.append(report)
            continue

        for col in ["bid", "ask", "lastPrice", "volume", "openInterest", "impliedVolatility"]:
            if col not in calls.columns:
                calls[col] = np.nan

        numeric_cols = [
            "strike",
            "bid",
            "ask",
            "lastPrice",
            "volume",
            "openInterest",
            "impliedVolatility",
        ]

        for col in numeric_cols:
            calls[col] = pd.to_numeric(calls[col], errors="coerce")

        calls = calls.dropna(subset=["strike"]).copy()
        calls["strike"] = calls["strike"].astype(float)
        calls["moneyness"] = calls["strike"] / S0

        calls = calls[
            (calls["moneyness"] >= cfg.moneyness_min)
            & (calls["moneyness"] <= cfg.moneyness_max)
        ].copy()

        report["moneyness"] = len(calls)

        if calls.empty:
            debug_rows.append(report)
            continue

        calls["mid"] = 0.5 * (calls["bid"] + calls["ask"])
        calls["spread"] = calls["ask"] - calls["bid"]
        calls["spread_pct"] = np.where(calls["mid"] > 0, calls["spread"] / calls["mid"], np.nan)

        valid_mid = (
            (calls["bid"] > 0)
            & (calls["ask"] > 0)
            & (calls["ask"] >= calls["bid"])
            & (calls["mid"] > 0)
        )

        if cfg.max_spread_pct is not None:
            valid_mid = valid_mid & (calls["spread_pct"] <= cfg.max_spread_pct)

        valid_last = calls["lastPrice"] > 0

        calls["valid_mid"] = valid_mid
        calls["valid_last"] = valid_last

        report["valid_mid"] = int(valid_mid.sum())
        report["valid_last"] = int(valid_last.sum())

        if cfg.price_source == "mid":
            calls = calls[calls["valid_mid"]].copy()
            calls["market_price"] = calls["mid"]
            calls["price_type"] = "mid"

        elif cfg.price_source == "last":
            calls = calls[calls["valid_last"]].copy()
            calls["market_price"] = calls["lastPrice"]
            calls["price_type"] = "last"

        else:
            calls = calls[calls["valid_mid"] | calls["valid_last"]].copy()
            calls["market_price"] = np.where(calls["valid_mid"], calls["mid"], calls["lastPrice"])
            calls["price_type"] = np.where(calls["valid_mid"], "mid", "last")

        calls = calls[np.isfinite(calls["market_price"]) & (calls["market_price"] > 0)].copy()

        report["chosen_price"] = len(calls)

        if calls.empty:
            debug_rows.append(report)
            continue

        if cfg.min_volume is not None:
            calls["volume"] = calls["volume"].fillna(0)
            calls = calls[calls["volume"] >= cfg.min_volume].copy()

        if cfg.min_open_interest is not None:
            calls["openInterest"] = calls["openInterest"].fillna(0)
            calls = calls[calls["openInterest"] >= cfg.min_open_interest].copy()

        report["after_liquidity"] = len(calls)

        if calls.empty:
            debug_rows.append(report)
            continue

        calls["abs_moneyness_gap"] = np.abs(calls["moneyness"] - 1.0)
        calls = calls.sort_values(["abs_moneyness_gap", "strike"]).head(cfg.max_strikes_per_expiry)
        calls = calls.sort_values("strike").reset_index(drop=True)

        report["selected"] = len(calls)

        for _, rr in calls.iterrows():
            rows.append(
                {
                    "ticker": cfg.ticker,
                    "expiry": exp,
                    "T": float(T),
                    "T_days": float(dte),
                    "strike": float(rr["strike"]),
                    "market_price": float(rr["market_price"]),
                    "price_type": str(rr["price_type"]),
                    "bid": float(rr["bid"]) if np.isfinite(rr["bid"]) else np.nan,
                    "ask": float(rr["ask"]) if np.isfinite(rr["ask"]) else np.nan,
                    "mid": float(rr["mid"]) if np.isfinite(rr["mid"]) else np.nan,
                    "lastPrice": float(rr["lastPrice"]) if np.isfinite(rr["lastPrice"]) else np.nan,
                    "spread": float(rr["spread"]) if np.isfinite(rr["spread"]) else np.nan,
                    "spread_pct": float(rr["spread_pct"]) if np.isfinite(rr["spread_pct"]) else np.nan,
                    "impliedVolatility": float(rr["impliedVolatility"])
                    if np.isfinite(rr["impliedVolatility"])
                    else np.nan,
                    "volume": float(rr["volume"]) if np.isfinite(rr["volume"]) else np.nan,
                    "openInterest": float(rr["openInterest"])
                    if np.isfinite(rr["openInterest"])
                    else np.nan,
                    "moneyness": float(rr["moneyness"]),
                }
            )

        debug_rows.append(report)

    debug_df = pd.DataFrame(debug_rows)

    if cfg.data_debug and not debug_df.empty:
        print("\n[yfinance option filter report]")
        print(debug_df.to_string(index=False))

    df = pd.DataFrame(rows)

    if df.empty:
        raise RuntimeError(
            "No option rows survived filtering. "
            "Try price_source='last', wider moneyness range, max_spread_pct=None, or larger max_dte."
        )

    df = df.sort_values(["T", "expiry", "strike"]).reset_index(drop=True)

    return S0, df


# ============================================================
# Train-validation split
# ============================================================

def split_by_expiry_alternating(
    df: pd.DataFrame,
    min_contracts_per_expiry: int = 4,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    train_parts = []
    val_parts = []

    df0 = df.sort_values(["T", "expiry", "strike"]).reset_index(drop=True)

    for _, sub in df0.groupby("expiry", sort=False):
        sub = sub.sort_values("strike").reset_index(drop=True)

        if len(sub) < min_contracts_per_expiry:
            continue

        train = sub.iloc[::2].copy()
        val = sub.iloc[1::2].copy()

        if len(train) == 0 or len(val) == 0:
            continue

        train_parts.append(train)
        val_parts.append(val)

    if not train_parts or not val_parts:
        raise RuntimeError(
            "Train-validation split failed. Increase max_strikes_per_expiry or loosen filters."
        )

    df_train = pd.concat(train_parts, ignore_index=True)
    df_val = pd.concat(val_parts, ignore_index=True)

    df_train = df_train.sort_values(["T", "expiry", "strike"]).reset_index(drop=True)
    df_val = df_val.sort_values(["T", "expiry", "strike"]).reset_index(drop=True)

    return df_train, df_val


# ============================================================
# Black-Scholes
# ============================================================

def bs_call_price(
    S0: float,
    K: float,
    T: float,
    r: float,
    sigma: float,
    q: float = 0.0,
) -> float:
    S0 = float(S0)
    K = float(K)
    T = float(T)
    r = float(r)
    sigma = float(sigma)
    q = float(q)

    if T <= 0:
        return max(S0 - K, 0.0)

    if S0 <= 0 or K <= 0:
        return np.nan

    if sigma <= 0 or not np.isfinite(sigma):
        return max(S0 * np.exp(-q * T) - K * np.exp(-r * T), 0.0)

    vol_sqrt = sigma * np.sqrt(T)

    d1 = (np.log(S0 / K) + (r - q + 0.5 * sigma * sigma) * T) / vol_sqrt
    d2 = d1 - vol_sqrt

    price = S0 * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

    return float(price)


def fit_bs_per_expiry(
    S0: float,
    df_cal: pd.DataFrame,
    r: float,
    q: float,
    sigma_grid: Iterable[float],
) -> pd.DataFrame:
    grid = _as_float_array(sigma_grid)
    grid = grid[grid > 0]

    if grid.size == 0:
        raise ValueError("sigma_grid is empty")

    rows = []

    for expiry, sub in df_cal.groupby("expiry", sort=False):
        T = float(sub["T"].iloc[0])
        Ks = sub["strike"].to_numpy(dtype=float)
        y = sub["market_price"].to_numpy(dtype=float)

        best_sigma = np.nan
        best_sse = np.inf

        for sigma in grid:
            pred = np.array([bs_call_price(S0, K, T, r, sigma, q=q) for K in Ks])

            if np.any(~np.isfinite(pred)):
                continue

            sse = float(np.sum((y - pred) ** 2))

            if sse < best_sse:
                best_sse = sse
                best_sigma = float(sigma)

        n = len(sub)

        rows.append(
            {
                "expiry": expiry,
                "T": T,
                "n_cal": n,
                "best_sigma_BS": best_sigma,
                "SSE_cal_BS": best_sse,
                "RMSE_cal_BS": float(np.sqrt(best_sse / n)) if np.isfinite(best_sse) else np.nan,
            }
        )

    return pd.DataFrame(rows).sort_values("T").reset_index(drop=True)


def fit_bs_coarse_to_fine(
    S0: float,
    df_cal: pd.DataFrame,
    r: float,
    q: float,
    sigma_grid_coarse: Iterable[float],
    sigma_radius: float,
    sigma_step: float,
    sigma_bounds: Tuple[float, float],
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    df_coarse = fit_bs_per_expiry(S0, df_cal, r, q, sigma_grid_coarse)

    fine_parts = []

    for _, row in df_coarse.iterrows():
        expiry = row["expiry"]
        sigma0 = float(row["best_sigma_BS"])
        sub = df_cal[df_cal["expiry"] == expiry]

        fine_grid = local_grid(
            center=sigma0,
            radius=sigma_radius,
            step=sigma_step,
            lower=sigma_bounds[0],
            upper=sigma_bounds[1],
        )

        if fine_grid.size == 0:
            fine_parts.append(pd.DataFrame([row]))
        else:
            fine_parts.append(fit_bs_per_expiry(S0, sub, r, q, fine_grid))

    df_fine = pd.concat(fine_parts, ignore_index=True)

    return df_fine.sort_values("T").reset_index(drop=True), df_coarse


def evaluate_bs_per_expiry(
    S0: float,
    df_val: pd.DataFrame,
    r: float,
    q: float,
    df_bs_fit: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    for _, fit_row in df_bs_fit.iterrows():
        expiry = fit_row["expiry"]
        sigma = float(fit_row["best_sigma_BS"])

        if not np.isfinite(sigma):
            continue

        sub = df_val[df_val["expiry"] == expiry]

        if sub.empty:
            continue

        T = float(sub["T"].iloc[0])
        Ks = sub["strike"].to_numpy(dtype=float)
        y = sub["market_price"].to_numpy(dtype=float)

        pred = np.array([bs_call_price(S0, K, T, r, sigma, q=q) for K in Ks])

        if np.any(~np.isfinite(pred)):
            continue

        sse = float(np.sum((y - pred) ** 2))
        n = len(sub)

        rows.append(
            {
                "expiry": expiry,
                "T": T,
                "n_val": n,
                "SSE_val_BS": sse,
                "RMSE_val_BS": float(np.sqrt(sse / n)),
            }
        )

    return pd.DataFrame(rows).sort_values("T").reset_index(drop=True)


# ============================================================
# TFBS pricer
# ============================================================

def caputo_tempered_weights(
    alpha: float,
    lam: float,
    N: int,
    dt: float,
) -> Tuple[np.ndarray, np.ndarray, float]:
    alpha = float(alpha)
    lam = float(lam)
    N = int(N)
    dt = float(dt)

    if not (0.0 < alpha < 1.0):
        raise ValueError("alpha must be in (0, 1)")
    if lam < 0:
        raise ValueError("lambda must be nonnegative")
    if N < 1:
        raise ValueError("N must be >= 1")
    if dt <= 0:
        raise ValueError("dt must be positive")

    k = np.arange(N + 1, dtype=float)

    w = (k + 1.0) ** (1.0 - alpha) - k ** (1.0 - alpha)
    damp = np.exp(-lam * k * dt)
    coef0 = 1.0 / ((dt ** alpha) * gamma(2.0 - alpha))

    return w, damp, float(coef0)


def build_bs_operator_coefficients(
    S: np.ndarray,
    r: float,
    q: float,
    sigma: float,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    dS = float(S[1] - S[0])
    Sj = S[1:-1]

    drift = r - q

    a = 0.5 * sigma * sigma * Sj * Sj / (dS * dS) - 0.5 * drift * Sj / dS
    b = -sigma * sigma * Sj * Sj / (dS * dS) - r
    c = 0.5 * sigma * sigma * Sj * Sj / (dS * dS) + 0.5 * drift * Sj / dS

    return a, b, c


def tfbs_call_price(
    S0: float,
    K: float,
    T: float,
    r: float,
    q: float,
    sigma: float,
    alpha: float,
    lam: float,
    M: int = 50,
    N: int = 50,
    smax_mult: float = 3.0,
    pre_weights: Optional[Tuple[np.ndarray, np.ndarray, float]] = None,
) -> float:
    S0 = float(S0)
    K = float(K)
    T = float(T)
    r = float(r)
    q = float(q)
    sigma = float(sigma)
    alpha = float(alpha)
    lam = float(lam)
    M = int(M)
    N = int(N)

    if T <= 0:
        return max(S0 - K, 0.0)

    if S0 <= 0 or K <= 0 or sigma <= 0:
        return np.nan

    if not (0.0 < alpha < 1.0):
        return np.nan

    if lam < 0:
        return np.nan

    if M < 3 or N < 1:
        raise ValueError("Need M >= 3 and N >= 1")

    Smax = float(smax_mult) * max(S0, K)
    S = np.linspace(0.0, Smax, M + 1)
    dt = T / N

    if pre_weights is None:
        w, damp, coef0 = caputo_tempered_weights(alpha, lam, N, dt)
    else:
        w, damp, coef0 = pre_weights

    a, b, c = build_bs_operator_coefficients(S, r, q, sigma)

    n_inner = M - 1

    diag = coef0 - b
    lower = -a[1:]
    upper = -c[:-1]

    ab = np.zeros((3, n_inner), dtype=float)
    ab[0, 1:] = upper
    ab[1, :] = diag
    ab[2, :-1] = lower

    C = np.zeros((M + 1, N + 1), dtype=float)

    C[:, 0] = np.maximum(S - K, 0.0)
    C[0, :] = 0.0

    for n in range(N):
        tau_next = (n + 1) * dt

        C_left = 0.0
        C_right = max(Smax * np.exp(-q * tau_next) - K * np.exp(-r * tau_next), 0.0)

        old_inner = C[1:M, n]

        if n == 0:
            hist = np.zeros(n_inner, dtype=float)
        else:
            diffs = C[1:M, 1 : n + 1] - C[1:M, 0:n]
            hist_weights = (w[1 : n + 1] * damp[1 : n + 1])[::-1]
            hist = np.sum(diffs * hist_weights[None, :], axis=1)

        rhs = coef0 * old_inner - coef0 * hist

        rhs[0] += a[0] * C_left
        rhs[-1] += c[-1] * C_right

        new_inner = solve_banded((1, 1), ab, rhs)

        C[0, n + 1] = C_left
        C[1:M, n + 1] = new_inner
        C[M, n + 1] = C_right

    return float(np.interp(S0, S, C[:, -1]))


# ============================================================
# TFBS fitting
# ============================================================

def fit_tfbs_per_expiry(
    S0: float,
    df_cal: pd.DataFrame,
    r: float,
    q: float,
    alpha: float,
    sigma_grid: Iterable[float],
    lambda_grid: Iterable[float],
    M: int,
    N: int,
    smax_mult: float,
) -> pd.DataFrame:
    sigma_grid_arr = _as_float_array(sigma_grid)
    sigma_grid_arr = sigma_grid_arr[sigma_grid_arr > 0]

    lambda_grid_arr = _as_float_array(lambda_grid)
    lambda_grid_arr = lambda_grid_arr[lambda_grid_arr >= 0]

    if sigma_grid_arr.size == 0:
        raise ValueError("sigma_grid is empty")
    if lambda_grid_arr.size == 0:
        raise ValueError("lambda_grid is empty")

    weights_cache: Dict[Tuple[float, float], Tuple[np.ndarray, np.ndarray, float]] = {}

    def get_weights(T: float, lam: float) -> Tuple[np.ndarray, np.ndarray, float]:
        key = (float(T), float(lam))

        if key not in weights_cache:
            dt = float(T) / int(N)
            weights_cache[key] = caputo_tempered_weights(alpha, lam, int(N), dt)

        return weights_cache[key]

    rows = []

    for expiry, sub in df_cal.groupby("expiry", sort=False):
        T = float(sub["T"].iloc[0])
        Ks = sub["strike"].to_numpy(dtype=float)
        y = sub["market_price"].to_numpy(dtype=float)

        best_sigma = np.nan
        best_lambda = np.nan
        best_sse = np.inf

        for lam in lambda_grid_arr:
            pre_weights = get_weights(T, float(lam))

            for sigma in sigma_grid_arr:
                pred = np.array(
                    [
                        tfbs_call_price(
                            S0=S0,
                            K=K,
                            T=T,
                            r=r,
                            q=q,
                            sigma=float(sigma),
                            alpha=float(alpha),
                            lam=float(lam),
                            M=M,
                            N=N,
                            smax_mult=smax_mult,
                            pre_weights=pre_weights,
                        )
                        for K in Ks
                    ],
                    dtype=float,
                )

                if np.any(~np.isfinite(pred)):
                    continue

                sse = float(np.sum((y - pred) ** 2))

                if sse < best_sse:
                    best_sse = sse
                    best_sigma = float(sigma)
                    best_lambda = float(lam)

        n = len(sub)

        rows.append(
            {
                "expiry": expiry,
                "T": T,
                "n_cal": n,
                "alpha": float(alpha),
                "best_sigma_TFBS": best_sigma,
                "best_lambda_TFBS": best_lambda,
                "SSE_cal_TFBS": best_sse,
                "RMSE_cal_TFBS": float(np.sqrt(best_sse / n)) if np.isfinite(best_sse) else np.nan,
            }
        )

    return pd.DataFrame(rows).sort_values("T").reset_index(drop=True)


def scan_alpha_on_calibration(
    S0: float,
    df_cal: pd.DataFrame,
    r: float,
    q: float,
    alpha_grid: Iterable[float],
    sigma_grid: Iterable[float],
    lambda_grid: Iterable[float],
    M: int,
    N: int,
    smax_mult: float,
    stage: str,
    progress: bool = True,
) -> pd.DataFrame:
    rows = []

    for alpha in _as_float_array(alpha_grid):
        if not (0.0 < float(alpha) < 1.0):
            continue

        if progress:
            print(f"[alpha scan:{stage}] alpha={alpha:.4f}")

        df_fit = fit_tfbs_per_expiry(
            S0=S0,
            df_cal=df_cal,
            r=r,
            q=q,
            alpha=float(alpha),
            sigma_grid=sigma_grid,
            lambda_grid=lambda_grid,
            M=M,
            N=N,
            smax_mult=smax_mult,
        )

        rows.append(
            {
                "stage": stage,
                "alpha": float(alpha),
                "pooled_cal_RMSE_TFBS": pooled_rmse_from_sse(
                    df_fit,
                    "SSE_cal_TFBS",
                    "n_cal",
                ),
                "n_expiries": int(df_fit["expiry"].nunique()),
                "n_cal_total": int(df_fit["n_cal"].sum()),
            }
        )

    if not rows:
        raise RuntimeError("alpha scan produced no rows")

    return pd.DataFrame(rows).sort_values("pooled_cal_RMSE_TFBS").reset_index(drop=True)


def fit_tfbs_coarse_to_fine(
    S0: float,
    df_cal: pd.DataFrame,
    r: float,
    q: float,
    alpha: float,
    sigma_grid_coarse: Iterable[float],
    lambda_grid_coarse: Iterable[float],
    M_coarse: int,
    N_coarse: int,
    M_fine: int,
    N_fine: int,
    smax_mult: float,
    sigma_radius: float,
    sigma_step: float,
    sigma_bounds: Tuple[float, float],
    lambda_radius: float,
    lambda_step: float,
    lambda_bounds: Tuple[float, float],
    progress: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    df_coarse = fit_tfbs_per_expiry(
        S0=S0,
        df_cal=df_cal,
        r=r,
        q=q,
        alpha=alpha,
        sigma_grid=sigma_grid_coarse,
        lambda_grid=lambda_grid_coarse,
        M=M_coarse,
        N=N_coarse,
        smax_mult=smax_mult,
    )

    fine_parts = []

    for _, row in df_coarse.iterrows():
        expiry = row["expiry"]
        sigma0 = float(row["best_sigma_TFBS"])
        lambda0 = float(row["best_lambda_TFBS"])

        sub = df_cal[df_cal["expiry"] == expiry]

        sigma_grid_fine = local_grid(
            sigma0,
            sigma_radius,
            sigma_step,
            sigma_bounds[0],
            sigma_bounds[1],
        )

        lambda_grid_fine = local_grid(
            lambda0,
            lambda_radius,
            lambda_step,
            lambda_bounds[0],
            lambda_bounds[1],
        )

        if progress:
            print(
                f"[TFBS local fine] {expiry}: "
                f"sigma0={sigma0:.4f}, lambda0={lambda0:.4f}, "
                f"grid={len(sigma_grid_fine)}x{len(lambda_grid_fine)}"
            )

        if sigma_grid_fine.size == 0 or lambda_grid_fine.size == 0:
            fine_parts.append(pd.DataFrame([row]))
        else:
            fine_parts.append(
                fit_tfbs_per_expiry(
                    S0=S0,
                    df_cal=sub,
                    r=r,
                    q=q,
                    alpha=alpha,
                    sigma_grid=sigma_grid_fine,
                    lambda_grid=lambda_grid_fine,
                    M=M_fine,
                    N=N_fine,
                    smax_mult=smax_mult,
                )
            )

    df_fine = pd.concat(fine_parts, ignore_index=True)

    return df_fine.sort_values("T").reset_index(drop=True), df_coarse


def evaluate_tfbs_per_expiry(
    S0: float,
    df_val: pd.DataFrame,
    r: float,
    q: float,
    df_tfbs_fit: pd.DataFrame,
    M: int,
    N: int,
    smax_mult: float,
) -> pd.DataFrame:
    weights_cache: Dict[Tuple[float, float, float], Tuple[np.ndarray, np.ndarray, float]] = {}

    def get_weights(T: float, alpha: float, lam: float) -> Tuple[np.ndarray, np.ndarray, float]:
        key = (float(T), float(alpha), float(lam))

        if key not in weights_cache:
            dt = float(T) / int(N)
            weights_cache[key] = caputo_tempered_weights(alpha, lam, int(N), dt)

        return weights_cache[key]

    rows = []

    for _, fit_row in df_tfbs_fit.iterrows():
        expiry = fit_row["expiry"]
        alpha = float(fit_row["alpha"])
        sigma = float(fit_row["best_sigma_TFBS"])
        lam = float(fit_row["best_lambda_TFBS"])

        if not (np.isfinite(alpha) and np.isfinite(sigma) and np.isfinite(lam)):
            continue

        sub = df_val[df_val["expiry"] == expiry]

        if sub.empty:
            continue

        T = float(sub["T"].iloc[0])
        Ks = sub["strike"].to_numpy(dtype=float)
        y = sub["market_price"].to_numpy(dtype=float)

        pre_weights = get_weights(T, alpha, lam)

        pred = np.array(
            [
                tfbs_call_price(
                    S0=S0,
                    K=K,
                    T=T,
                    r=r,
                    q=q,
                    sigma=sigma,
                    alpha=alpha,
                    lam=lam,
                    M=M,
                    N=N,
                    smax_mult=smax_mult,
                    pre_weights=pre_weights,
                )
                for K in Ks
            ],
            dtype=float,
        )

        if np.any(~np.isfinite(pred)):
            continue

        sse = float(np.sum((y - pred) ** 2))
        n = len(sub)

        rows.append(
            {
                "expiry": expiry,
                "T": T,
                "n_val": n,
                "SSE_val_TFBS": sse,
                "RMSE_val_TFBS": float(np.sqrt(sse / n)),
            }
        )

    return pd.DataFrame(rows).sort_values("T").reset_index(drop=True)


# ============================================================
# Driver / summary / plotting
# ============================================================

def summarize_validation_by_bucket(
    df_compare: pd.DataFrame,
    short_days: int,
    medium_days: int,
) -> pd.DataFrame:
    if df_compare.empty:
        return pd.DataFrame()

    df0 = add_maturity_bucket(
        df_compare,
        short_days=short_days,
        medium_days=medium_days,
    )

    rows = []

    for bucket in ["Short", "Medium", "Long"]:
        sub = df0[df0["bucket"] == bucket]

        if sub.empty:
            continue

        n = int(sub["n_val"].sum())

        if n <= 0:
            continue

        rmse_bs = float(np.sqrt(sub["SSE_val_BS"].sum() / n))
        rmse_tfbs = float(np.sqrt(sub["SSE_val_TFBS"].sum() / n))

        rows.append(
            {
                "bucket": bucket,
                "n_expiries": int(sub["expiry"].nunique()),
                "n_val": n,
                "RMSE_val_BS": rmse_bs,
                "RMSE_val_TFBS": rmse_tfbs,
                "improvement_pct": 100.0 * (rmse_bs - rmse_tfbs) / rmse_bs
                if rmse_bs > 0
                else np.nan,
            }
        )

    n_all = int(df0["n_val"].sum())

    if n_all > 0:
        rmse_bs = float(np.sqrt(df0["SSE_val_BS"].sum() / n_all))
        rmse_tfbs = float(np.sqrt(df0["SSE_val_TFBS"].sum() / n_all))

        rows.append(
            {
                "bucket": "All",
                "n_expiries": int(df0["expiry"].nunique()),
                "n_val": n_all,
                "RMSE_val_BS": rmse_bs,
                "RMSE_val_TFBS": rmse_tfbs,
                "improvement_pct": 100.0 * (rmse_bs - rmse_tfbs) / rmse_bs
                if rmse_bs > 0
                else np.nan,
            }
        )

    return pd.DataFrame(rows)


def run_tfbs_experiment(
    market_cfg: MarketDataConfig,
    exp_cfg: ExperimentConfig,
    df_options: Optional[pd.DataFrame] = None,
    S0: Optional[float] = None,
) -> Dict[str, Any]:
    if df_options is None:
        S0_used, df_all = collect_call_options_yfinance(market_cfg)
    else:
        if S0 is None:
            raise ValueError("S0 must be provided when df_options is provided")

        S0_used = float(S0)
        df_all = df_options.copy()

    required_cols = {"expiry", "T", "strike", "market_price"}
    missing = required_cols.difference(df_all.columns)

    if missing:
        raise ValueError(f"df_options missing columns: {sorted(missing)}")

    df_all = df_all.sort_values(["T", "expiry", "strike"]).reset_index(drop=True)
    df_train, df_val = split_by_expiry_alternating(df_all)

    if exp_cfg.progress:
        print("=" * 72)
        print(f"ticker={market_cfg.ticker}, S0={S0_used:.4f}")
        print(f"rows: all={len(df_all)}, calibration={len(df_train)}, validation={len(df_val)}")
        print(f"expiries used={df_all['expiry'].nunique()}")
        print("=" * 72)

    sigma_grid_coarse = np.asarray(exp_cfg.sigma_grid_coarse, dtype=float)
    lambda_grid_coarse = np.asarray(exp_cfg.lambda_grid_coarse, dtype=float)
    alpha_grid_coarse = np.asarray(exp_cfg.alpha_grid_coarse, dtype=float)

    # 1. BS
    df_bs_fit, df_bs_coarse = fit_bs_coarse_to_fine(
        S0=S0_used,
        df_cal=df_train,
        r=exp_cfg.r,
        q=exp_cfg.q,
        sigma_grid_coarse=sigma_grid_coarse,
        sigma_radius=exp_cfg.sigma_radius,
        sigma_step=exp_cfg.sigma_step,
        sigma_bounds=exp_cfg.sigma_bounds,
    )

    df_bs_val = evaluate_bs_per_expiry(
        S0=S0_used,
        df_val=df_val,
        r=exp_cfg.r,
        q=exp_cfg.q,
        df_bs_fit=df_bs_fit,
    )

    pooled_val_rmse_bs = pooled_rmse_from_sse(df_bs_val, "SSE_val_BS", "n_val")

    # 2. Alpha scan
    df_alpha_coarse = scan_alpha_on_calibration(
        S0=S0_used,
        df_cal=df_train,
        r=exp_cfg.r,
        q=exp_cfg.q,
        alpha_grid=alpha_grid_coarse,
        sigma_grid=sigma_grid_coarse,
        lambda_grid=lambda_grid_coarse,
        M=exp_cfg.m_scan,
        N=exp_cfg.n_scan,
        smax_mult=exp_cfg.smax_mult,
        stage="coarse",
        progress=exp_cfg.progress,
    )

    best_alpha_coarse = float(df_alpha_coarse.iloc[0]["alpha"])

    if exp_cfg.refine_alpha:
        alpha_grid_local = local_grid(
            center=best_alpha_coarse,
            radius=exp_cfg.alpha_radius,
            step=exp_cfg.alpha_step,
            lower=0.05,
            upper=0.99,
        )

        df_alpha_local = scan_alpha_on_calibration(
            S0=S0_used,
            df_cal=df_train,
            r=exp_cfg.r,
            q=exp_cfg.q,
            alpha_grid=alpha_grid_local,
            sigma_grid=sigma_grid_coarse,
            lambda_grid=lambda_grid_coarse,
            M=exp_cfg.m_scan,
            N=exp_cfg.n_scan,
            smax_mult=exp_cfg.smax_mult,
            stage="local",
            progress=exp_cfg.progress,
        )

        df_alpha_scan = pd.concat([df_alpha_coarse, df_alpha_local], ignore_index=True)
        best_alpha = float(df_alpha_local.iloc[0]["alpha"])

    else:
        alpha_grid_local = np.array([], dtype=float)
        df_alpha_scan = df_alpha_coarse.copy()
        best_alpha = best_alpha_coarse

    if exp_cfg.progress:
        print("-" * 72)
        print(f"best_alpha_coarse = {best_alpha_coarse:.4f}")
        print(f"best_alpha_final  = {best_alpha:.4f}")
        print("-" * 72)

    # 3. Final TFBS coarse-to-fine
    df_tfbs_fit, df_tfbs_coarse = fit_tfbs_coarse_to_fine(
        S0=S0_used,
        df_cal=df_train,
        r=exp_cfg.r,
        q=exp_cfg.q,
        alpha=best_alpha,
        sigma_grid_coarse=sigma_grid_coarse,
        lambda_grid_coarse=lambda_grid_coarse,
        M_coarse=exp_cfg.m_final_coarse,
        N_coarse=exp_cfg.n_final_coarse,
        M_fine=exp_cfg.m_final,
        N_fine=exp_cfg.n_final,
        smax_mult=exp_cfg.smax_mult,
        sigma_radius=exp_cfg.sigma_radius,
        sigma_step=exp_cfg.sigma_step,
        sigma_bounds=exp_cfg.sigma_bounds,
        lambda_radius=exp_cfg.lambda_radius,
        lambda_step=exp_cfg.lambda_step,
        lambda_bounds=exp_cfg.lambda_bounds,
        progress=exp_cfg.progress,
    )

    df_tfbs_val = evaluate_tfbs_per_expiry(
        S0=S0_used,
        df_val=df_val,
        r=exp_cfg.r,
        q=exp_cfg.q,
        df_tfbs_fit=df_tfbs_fit,
        M=exp_cfg.m_final,
        N=exp_cfg.n_final,
        smax_mult=exp_cfg.smax_mult,
    )

    pooled_val_rmse_tfbs = pooled_rmse_from_sse(
        df_tfbs_val,
        "SSE_val_TFBS",
        "n_val",
    )

    # 4. Merge
    df_compare = pd.merge(
        df_bs_fit,
        df_tfbs_fit,
        on=["expiry", "T", "n_cal"],
        how="inner",
    )

    df_compare = pd.merge(
        df_compare,
        df_bs_val,
        on=["expiry", "T"],
        how="inner",
    )

    df_compare = pd.merge(
        df_compare,
        df_tfbs_val,
        on=["expiry", "T", "n_val"],
        how="inner",
    )

    df_compare = add_maturity_bucket(
        df_compare,
        short_days=exp_cfg.short_days,
        medium_days=exp_cfg.medium_days,
    )

    df_compare["RMSE_val_gain"] = df_compare["RMSE_val_BS"] - df_compare["RMSE_val_TFBS"]
    df_compare["RMSE_val_improvement_pct"] = (
        100.0 * df_compare["RMSE_val_gain"] / df_compare["RMSE_val_BS"]
    )

    df_compare = df_compare.sort_values("T").reset_index(drop=True)

    df_summary = summarize_validation_by_bucket(
        df_compare,
        short_days=exp_cfg.short_days,
        medium_days=exp_cfg.medium_days,
    )

    if exp_cfg.progress:
        print("=" * 72)
        print(f"Validation RMSE | BS   = {pooled_val_rmse_bs:.6f}")
        print(f"Validation RMSE | TFBS = {pooled_val_rmse_tfbs:.6f}")
        print("=" * 72)

    return {
        "S0": S0_used,
        "market_cfg": market_cfg,
        "exp_cfg": exp_cfg,
        "best_alpha_coarse": best_alpha_coarse,
        "best_alpha": best_alpha,
        "alpha_grid_local": alpha_grid_local,
        "df_all": df_all,
        "df_train": df_train,
        "df_val": df_val,
        "df_alpha_scan": df_alpha_scan.sort_values(
            ["stage", "pooled_cal_RMSE_TFBS"]
        ).reset_index(drop=True),
        "df_bs_coarse": df_bs_coarse,
        "df_bs_fit": df_bs_fit,
        "df_bs_val": df_bs_val,
        "df_tfbs_coarse": df_tfbs_coarse,
        "df_tfbs_fit": df_tfbs_fit,
        "df_tfbs_val": df_tfbs_val,
        "df_compare_expiry": df_compare,
        "df_validation_summary": df_summary,
        "pooled_val_rmse_bs": pooled_val_rmse_bs,
        "pooled_val_rmse_tfbs": pooled_val_rmse_tfbs,
    }


def plot_results(res: Dict[str, Any]) -> None:
    df = res["df_compare_expiry"].sort_values("T").reset_index(drop=True)

    if df.empty:
        print("No expiry comparison data to plot.")
        return

    exp_cfg: ExperimentConfig = res["exp_cfg"]
    market_cfg: MarketDataConfig = res["market_cfg"]

    T = df["T"].to_numpy(dtype=float)
    t_short = exp_cfg.short_days / 365.25
    t_medium = exp_cfg.medium_days / 365.25

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.axvspan(t_short, t_medium, alpha=0.12, label="Medium bucket")
    ax.plot(T, df["RMSE_val_BS"], marker="o", label="BS validation RMSE")
    ax.plot(T, df["RMSE_val_TFBS"], marker="s", label="TFBS validation RMSE")
    ax.set_xlabel("T (years)")
    ax.set_ylabel("Validation RMSE")
    ax.set_title(
        f"{market_cfg.ticker}: validation pricing error\n"
        f"TFBS={res['pooled_val_rmse_tfbs']:.4f}, BS={res['pooled_val_rmse_bs']:.4f}"
    )
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.axvspan(t_short, t_medium, alpha=0.12, label="Medium bucket")
    ax.plot(T, df["best_sigma_BS"], marker="o", label="BS sigma")
    ax.plot(T, df["best_sigma_TFBS"], marker="s", label="TFBS sigma")
    ax.set_xlabel("T (years)")
    ax.set_ylabel("sigma")
    ax.set_title(f"{market_cfg.ticker}: volatility term structure")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.axvspan(t_short, t_medium, alpha=0.12, label="Medium bucket")
    ax.plot(T, df["best_lambda_TFBS"], marker="^", label="TFBS lambda")
    ax.set_xlabel("T (years)")
    ax.set_ylabel("lambda")
    ax.set_title(f"{market_cfg.ticker}: tempering term structure, alpha={res['best_alpha']:.3f}")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()


def diagnose_result(res: Dict[str, Any]) -> None:
    print("\n=== Overall ===")
    print("S0:", res["S0"])
    print("best_alpha_coarse:", res["best_alpha_coarse"])
    print("best_alpha:", res["best_alpha"])
    print("Validation RMSE | BS  :", res["pooled_val_rmse_bs"])
    print("Validation RMSE | TFBS:", res["pooled_val_rmse_tfbs"])

    print("\n=== Bucket summary ===")
    print(res["df_validation_summary"])

    print("\n=== Alpha scan ===")
    print(res["df_alpha_scan"])

    print("\n=== Expiry comparison ===")
    cols = [
        "expiry",
        "T_days",
        "bucket",
        "best_sigma_BS",
        "best_sigma_TFBS",
        "best_lambda_TFBS",
        "RMSE_val_BS",
        "RMSE_val_TFBS",
        "RMSE_val_improvement_pct",
    ]
    print(res["df_compare_expiry"][cols])

    if "price_type" in res["df_all"].columns:
        print("\n=== Price source counts ===")
        print(res["df_all"]["price_type"].value_counts(dropna=False))


def save_result_csvs(res: Dict[str, Any], prefix: Optional[str] = None) -> None:
    market_cfg: MarketDataConfig = res["market_cfg"]

    if prefix is None:
        prefix = f"{market_cfg.ticker}_tfbs"

    tables = {
        "all_options": res["df_all"],
        "train_options": res["df_train"],
        "validation_options": res["df_val"],
        "alpha_scan": res["df_alpha_scan"],
        "expiry_compare": res["df_compare_expiry"],
        "validation_summary": res["df_validation_summary"],
        "bs_fit": res["df_bs_fit"],
        "tfbs_fit": res["df_tfbs_fit"],
    }

    for name, df in tables.items():
        df.to_csv(f"{prefix}_{name}.csv", index=False)

    print(f"Saved CSV files with prefix: {prefix}_*.csv")

In [ ]:
# ============================================================
# 3A. Minimal execution helper
# Run this cell once.
# After this, you only edit TICKER and GRID in the last cell.
# ============================================================

def _tup(x):
    return tuple(float(v) for v in x)


GRID_PRESETS = {
    "smoke": {
        "alpha_grid_coarse": _tup([0.50, 0.75]),
        "sigma_grid_coarse": _tup([0.20, 0.30, 0.40]),
        "lambda_grid_coarse": _tup([0.0, 4.0, 8.0]),

        "m_scan": 25,
        "n_scan": 25,
        "m_final_coarse": 35,
        "n_final_coarse": 35,
        "m_final": 50,
        "n_final": 50,

        "refine_alpha": True,
        "alpha_radius": 0.10,
        "alpha_step": 0.05,

        "sigma_radius": 0.05,
        "sigma_step": 0.025,

        "lambda_radius": 4.0,
        "lambda_step": 2.0,
    },

    "coarse": {
        "alpha_grid_coarse": _tup([0.50, 0.65, 0.75, 0.85, 0.95]),
        "sigma_grid_coarse": _tup([0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]),
        "lambda_grid_coarse": _tup([0.0, 2.0, 4.0, 6.0, 8.0, 10.0, 12.0]),

        "m_scan": 30,
        "n_scan": 30,
        "m_final_coarse": 45,
        "n_final_coarse": 45,
        "m_final": 70,
        "n_final": 70,

        "refine_alpha": True,
        "alpha_radius": 0.10,
        "alpha_step": 0.025,

        "sigma_radius": 0.05,
        "sigma_step": 0.02,

        "lambda_radius": 4.0,
        "lambda_step": 1.0,
    },

    "medium": {
        "alpha_grid_coarse": _tup(np.arange(0.45, 0.96, 0.05)),
        "sigma_grid_coarse": _tup(np.arange(0.10, 0.51, 0.05)),
        "lambda_grid_coarse": _tup(np.arange(0.0, 16.1, 2.0)),

        "m_scan": 40,
        "n_scan": 40,
        "m_final_coarse": 60,
        "n_final_coarse": 60,
        "m_final": 90,
        "n_final": 90,

        "refine_alpha": True,
        "alpha_radius": 0.075,
        "alpha_step": 0.025,

        "sigma_radius": 0.05,
        "sigma_step": 0.01,

        "lambda_radius": 3.0,
        "lambda_step": 0.5,
    },

    "dense": {
        "alpha_grid_coarse": _tup(np.arange(0.40, 0.96, 0.05)),
        "sigma_grid_coarse": _tup(np.arange(0.08, 0.61, 0.04)),
        "lambda_grid_coarse": _tup(np.arange(0.0, 20.1, 2.0)),

        "m_scan": 50,
        "n_scan": 50,
        "m_final_coarse": 80,
        "n_final_coarse": 80,
        "m_final": 120,
        "n_final": 120,

        "refine_alpha": True,
        "alpha_radius": 0.075,
        "alpha_step": 0.0125,

        "sigma_radius": 0.04,
        "sigma_step": 0.005,

        "lambda_radius": 3.0,
        "lambda_step": 0.5,
    },
}


def _make_profile_from_grid(GRID):
    """
    GRID can be either:

    1. preset name:
        GRID = "smoke"
        GRID = "coarse"
        GRID = "medium"
        GRID = "dense"

    2. custom grid dict:
        GRID = {
            "alpha": [0.50, 0.75],
            "sigma": [0.20, 0.30, 0.40],
            "lambda": [0, 4, 8],
        }

    Custom grid uses smoke-level PDE resolution by default.
    """
    if isinstance(GRID, str):
        if GRID not in GRID_PRESETS:
            raise ValueError(f"Unknown GRID preset: {GRID}")
        return GRID_PRESETS[GRID].copy(), GRID

    if isinstance(GRID, dict):
        profile = GRID_PRESETS["smoke"].copy()

        alpha_grid = GRID.get("alpha", GRID.get("alpha_grid_coarse"))
        sigma_grid = GRID.get("sigma", GRID.get("sigma_grid_coarse"))
        lambda_grid = GRID.get("lambda", GRID.get("lam", GRID.get("lambda_grid_coarse")))

        if alpha_grid is None:
            raise ValueError("Custom GRID must contain 'alpha'")
        if sigma_grid is None:
            raise ValueError("Custom GRID must contain 'sigma'")
        if lambda_grid is None:
            raise ValueError("Custom GRID must contain 'lambda'")

        profile["alpha_grid_coarse"] = _tup(alpha_grid)
        profile["sigma_grid_coarse"] = _tup(sigma_grid)
        profile["lambda_grid_coarse"] = _tup(lambda_grid)

        # Optional overrides are allowed but not required.
        optional_keys = [
            "m_scan",
            "n_scan",
            "m_final_coarse",
            "n_final_coarse",
            "m_final",
            "n_final",
            "refine_alpha",
            "alpha_radius",
            "alpha_step",
            "sigma_radius",
            "sigma_step",
            "lambda_radius",
            "lambda_step",
        ]

        for key in optional_keys:
            if key in GRID:
                profile[key] = GRID[key]

        return profile, "custom"

    raise TypeError("GRID must be either a preset string or a custom grid dict.")


def run_simple_tfbs(TICKER, GRID):
    """
    Minimal user-facing runner.

    User only controls:
        TICKER
        GRID

    Everything else is fixed here.
    """
    if isinstance(TICKER, str):
        tickers = [TICKER]
    else:
        tickers = list(TICKER)

    profile, grid_name = _make_profile_from_grid(GRID)

    results = {}
    summary_tables = []
    expiry_tables = []

    print("=" * 80)
    print(f"TICKER(S): {tickers}")
    print(f"GRID     : {grid_name}")
    print("=" * 80)

    print("Grid details:")
    print("alpha :", profile["alpha_grid_coarse"])
    print("sigma :", profile["sigma_grid_coarse"])
    print("lambda:", profile["lambda_grid_coarse"])
    print("=" * 80)

    for ticker in tickers:
        print("\n" + "#" * 80)
        print(f"Running {ticker}")
        print("#" * 80)

        market_cfg = MarketDataConfig(
            ticker=ticker,
            period="1y",

            # yfinance에서 bid/ask가 0으로 자주 나오므로 기본은 last.
            price_source="last",

            # Short / Medium / Long이 골고루 들어오게 DTE target 지정.
            target_dtes=(14, 30, 60, 90, 120, 150, 180, 210, 270, 360),
            max_expiries=None,

            max_strikes_per_expiry=8,

            min_dte=0,
            max_dte=365,

            moneyness_min=0.70,
            moneyness_max=1.30,

            max_spread_pct=None,

            min_volume=None,
            min_open_interest=None,

            data_debug=True,
        )

        exp_cfg = ExperimentConfig(
            r=0.0426,
            q=0.0,

            alpha_grid_coarse=profile["alpha_grid_coarse"],
            sigma_grid_coarse=profile["sigma_grid_coarse"],
            lambda_grid_coarse=profile["lambda_grid_coarse"],

            m_scan=profile["m_scan"],
            n_scan=profile["n_scan"],

            m_final_coarse=profile["m_final_coarse"],
            n_final_coarse=profile["n_final_coarse"],
            m_final=profile["m_final"],
            n_final=profile["n_final"],

            refine_alpha=profile["refine_alpha"],
            alpha_radius=profile["alpha_radius"],
            alpha_step=profile["alpha_step"],

            sigma_radius=profile["sigma_radius"],
            sigma_step=profile["sigma_step"],

            lambda_radius=profile["lambda_radius"],
            lambda_step=profile["lambda_step"],

            short_days=30,
            medium_days=180,

            progress=True,
        )

        try:
            res = run_tfbs_experiment(market_cfg, exp_cfg)
            results[ticker] = res

            diagnose_result(res)
            plot_results(res)

            summary = res["df_validation_summary"].copy()
            summary.insert(0, "ticker", ticker)
            summary.insert(1, "grid", grid_name)
            summary_tables.append(summary)

            expiry = res["df_compare_expiry"].copy()
            expiry.insert(0, "ticker", ticker)
            expiry.insert(1, "grid", grid_name)
            expiry_tables.append(expiry)

        except Exception as exc:
            print(f"[FAILED] {ticker}: {type(exc).__name__}: {exc}")

    if summary_tables:
        summary_all = pd.concat(summary_tables, ignore_index=True)
    else:
        summary_all = pd.DataFrame()

    if expiry_tables:
        expiry_all = pd.concat(expiry_tables, ignore_index=True)
    else:
        expiry_all = pd.DataFrame()

    print("\n" + "=" * 80)
    print("Combined bucket summary")
    print("=" * 80)

    if not summary_all.empty:
        display(summary_all)
    else:
        print("No summary table.")

    print("\n" + "=" * 80)
    print("Combined expiry-level comparison")
    print("=" * 80)

    if not expiry_all.empty:
        cols = [
            "ticker",
            "grid",
            "expiry",
            "T_days",
            "bucket",
            "best_sigma_BS",
            "best_sigma_TFBS",
            "best_lambda_TFBS",
            "RMSE_val_BS",
            "RMSE_val_TFBS",
            "RMSE_val_improvement_pct",
        ]

        cols = [c for c in cols if c in expiry_all.columns]
        display(expiry_all[cols])
    else:
        print("No expiry table.")

    return results, summary_all, expiry_all

In [5]:
# ============================================================
# Commodity / spot-linked ETFs
# GLD + USO
# ============================================================

TICKER = ["GOOGL", "AMZN", "JPM"]

GRID = {
    "alpha": [0.35, 0.50, 0.65, 0.75, 0.85, 0.95],
    "sigma": [0.08, 0.15, 0.25, 0.35, 0.50, 0.65, 0.80],
    "lambda": [0.0, 2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 16.0, 20.0],

    "m_scan": 30,
    "n_scan": 30,

    "m_final_coarse": 45,
    "n_final_coarse": 45,
    "m_final": 70,
    "n_final": 70,

    "refine_alpha": True,
    "alpha_radius": 0.10,
    "alpha_step": 0.025,

    "sigma_radius": 0.06,
    "sigma_step": 0.015,

    "lambda_radius": 4.0,
    "lambda_step": 1.0,
}

results_comm, summary_comm, expiry_comm = run_simple_tfbs(TICKER, GRID)

NameError: name 'run_simple_tfbs' is not defined


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\ws20w\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
